# Generic Single Point

<div class="alert alert-info">
<strong>Exercise: Subset a global dataset to run a generic single point case</strong><br><br>

1) Subset our global datasets to run for a single point

2) Create and run a case called **Boulder_bgc.generic** using the:
  - compset `I1PtClm60BgcRsGs`, the 
  - resolution `CLM_USRDAT`, and that 
  - applies `USERMODS` to use the subset data

In this tutorial we'll use the `subset_data` script to create a surface dataset and data atmosphere (datm) for a particular point.  Then we'll create a case using the CTSM workflow with the usermods created by `subset_data`.
</div>

## Background

### `subset_data`

<div class="alert alert-info">
<strong>Exercise </strong><br>

- Activate `ctsm_pylib` and 
- Move into the `tools/site_and_regional` directory:

```shell
conda activate ctsm_pylib
cd /glade/u/home/$USER/code/my_clm_code/tools/site_and_regional
```

- Look at the arguments subset_data requires for single point applications:

```shell
./subset_data point --help
```

You should see lots of arguments, starting with:
```
  --lat PLAT            Single point latitude. [default: 42.5]
  --lon PLON            Single point longitude. [default: 287.8]
  --lon-type {180,360}  Whether longitudes are in the [-180, 180] format (centered around the
                        Prime/0th Meridian) or the [0, 360] format (centered around the 180th
                        Meridian). Choose by specifying the upper limit.
  --site SITE_NAME      Site name or tag. [default: ]
  --uniform-snowpack    Modify surface data to have a uniform snow fraction.
  --include-nonveg      Do not zero non-vegetation land units in the surface data.
  --cap-saturation      Modify surface data to not allow saturation excess.
  --evenly_split_cropland
                        Introduce equal areas of all crops
  --dompft [DOM_PFT ...]
                        Dominant PFT(s): if we set the grid to 100% one or multiple PFTs
                        [default: None]
  ....
```

The `subset_data` script provides lots of ways to customize your surface dataset! 
First let's create a place to store your data and usermods

```shell
mkdir ~/boulder_point_data
```

The example that follows highlights some of the common ways you can do this.  Specifically in this example we'll create a:
- surface dataset for a single point case centered around Boulder, Colorado that assumes 100% temperate needleleaf evergreen trees and uniform snow coverage across the gridcell.
   - you can find a list of PFTs in the [CLM tech note](https://escomp.github.io/CTSM/tech_note/Photosynthetic_Capacity/CLM50_Tech_Note_Photosynthetic_Capacity.html#id5)
- Data atmosphere subset from CRUJRA for the year 2020
- usermods that can point to the data we created.

```shell
./subset_data point --lat 40.015 --lon -105.27 \
                    --lon-type 180 --site Boulder \
                    --uniform-snowpack --dompft 1 \
                    --create-surface  \
                    --create-datm  \
                    --datm-syr 2020 --datm-eyr 2020 \
                    --create-user-mods \
                    --outdir ~/boulder_point_data \
                    --user-mods-dir ~/boulder_point_data/usermods                    
```

Look at the files created:

```shell
ls -1 ~/boulder_point_data
```

expected output includes:

```shell
datmdata
surfdata_Boulder_hist_2000_16pfts_c261014.nc
usermods
```

**Check Your Understanding**

What does your datm data look like?

What does your surface data look like?

*Hint: use `ncdump -h`*

</div>

### Usermods

A **usermod directory** (“user modifications” directory) is a bundle of case-specific settings that CLM/CIME can apply when you create or clone a case. It’s a convenient way to start from a known configuration rather than manually setting every option afterward. The usermods may include files such as CLM namelist settings (user_nl_clm), settings for other model components, and XML configuration changes. 

You pass the bundle to CIME when creating a case. For example:
```
./create_newcase ... --user-mods-dir ~/boulder_point_data/usermods
```

<div class="alert alert-info">
<strong>Exercise </strong><br>

You can look to see what usermods settings your Boulder cases will inherit. 
```
ls -1 ~/boulder_point_data/usermods
```

expected output includes:
```
shell_commands
user_nl_clm
user_nl_datm_streams
```

**Check Your Understanding**

How are Boulder site-level details passed to CLM? 

*Hint: look at files in the usermod directory*

</div>

## Get Started
Let's create a BGC case for a generic site in Boulder.


<div class="alert alert-warning">  

<summary> <font color='black' size=+2> Overview </font> </summary><br>
    
**Create a new case**

You can create a new case from the `cime/scripts/` directory with the command:

```shell
./create_newcase --case /glade/u/home/$USER/cases/CASE --res RES --compset COMPSET --user-mods-dir USERMODS --run-unsupported 
```

Make sure you use the correct information for the CASE, RES, COMPSET, and USERMODS inputs.

**How do you customize your case to run a BGC case for this experiment?**

Specifically you'll want to make namelist changes that:
- set up a BGC case that points to your newly created USERMODS directory.
- modify history file output to generate some daily data for ths single point case.

And XML changes to run:
- We'll use initial conditions that have been provided for our BGC case. *This may not be a good choice, but it's quicker*;
- Run a quick spinup to initialize the soil moisture state (would 4 years should be enough?);
- Run a one year production run

**Did you set up the case?**

You can invoke case set up as follows:

```shell
./case.setup
```

**How do I check my case**

You can check your CaseDocs (especially lnd_in) after you run:

```shell
./preview_namelists
```

**How do I compile?**

You can compile with the command:

```shell
qcmd -- ./case.build
```

**How do I check my job status?**

You can check job status using the command:

```shell
qstat -u $USER
```    
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue' size=+1>
    Click here for the solution
</font></summary><br>
                
**Create a new case <font face="Courier" color='purple'><strong>Boulder_bgc.generic </strong></font>**

```shell
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts
```

Then create a new case:

```shell
./create_newcase --case /glade/u/home/$USER/cases/Boulder_bgc.generic \
    --res CLM_USRDAT \
    --compset I1PtClm60BgcRsGs \
    --run-unsupported \
    --user-mods-dir /glade/u/home/$USER/boulder_point_data/usermods 
```

**Case <font face="Courier" color='purple'><strong>setup</strong></font>**

```shell
cd /glade/u/home/$USER/cases/Boulder_bgc.generic
./case.setup
```

**Case <font face="Courier" color='purple'><strong>customization</strong></font>**

Make **user_nl_clm** changes in the single-point compset we chose above by adding the following into `user_nl_clm`:

```shell
echo "! Monthly means in h0a (n=1), Daily means in h1a files (n=365)" >> user_nl_clm
echo "hist_nhtfrq = 0, -24" >> user_nl_clm
echo "hist_mfilt  = 1, 365" >> user_nl_clm
echo "hist_fincl2 = 'FCEV', 'FCTR', 'FGEV', 'FIRA', 'FSA', 'FSH', 'GPP', 'ELAI', 'H2OSOI', 'SNOW_DEPTH', 'TBOT', 'TSOI'" >> user_nl_clm
```

These changes will provide:
- Monthly means in `h0a` files (time dimension; n = 1)
- Daily mean files `h1a` files (time dimension; n = 365).

**XML changes**

First look and see what default settings are for:
- Running from a cold start (without initial conditions)
- What years of DATM data do we have?
- How long are we running the case?

```shell
./xmlquery CLM_FORCE_COLDSTART
./xmlquery DATM_YR_START DATM_YR_ALIGN DATM_YR_END
./xmlquery STOP_OPTION STOP_N 
```

Expected results are:

```shell
	CLM_FORCE_COLDSTART: off

	DATM_YR_START: 2020
	DATM_YR_ALIGN: 2020
	DATM_YR_END: 2020

	STOP_OPTION: ndays
	STOP_N: 5

```

Now we'll change this to:
- Cycle over the one year DATM data available five times 
- Start the simulation in 2016, then we'll have 4 years of spinup and 1 year of our production run.
- **NOTE** in reality a longer spinup would be needed for this BGC case.

```shell
./xmlchange CLM_FORCE_COLDSTART=off
./xmlchange STOP_OPTION=nyears
./xmlchange STOP_N=5  
./xmlchange RUN_STARTDATE=2016-01-01
./xmlchange DATM_MODE=CLMCRUJRA2024b
```

**<font face="Courier" color='purple'><strong>Preview namelist</strong></font>**

Run preview_namelists (`./preview_namelists`) and check our variables:

Check that your lnd_in file looks OK, e.g.:
- `finidat = '/glade/campaign/cesm/cesmdata/inputdata/lnd/clm2/initdata_esmf/ctsm5.4/...'`
- `fsurdat = *surfdata_Boulder_hist_2000_16pfts*`
- `use_fates = .false.`
- `use_crop = .false.`

```shell
head -50 CaseDocs/lnd_in
```

**Case <font face="Courier" color='purple'><strong>build</strong></font>**

```shell
qcmd -- ./case.build 
```

**Case <font face="Courier" color='purple'><strong>submit</strong></font>**

```shell
./case.submit
```

Check the status:

```shell
qstat -u $USER
```    

When the run is completed, look into the archive directory for: 
<font face="Courier" color='purple'><strong>Boulder_bgc.generic</strong></font>.  
    
Check that your archive directory on derecho (the path will be different on other machines): 

```shell
cd /glade/derecho/scratch/$USER/archive/Boulder_bgc.generic
ls 
```

</details>
</div>


## Test your understanding

- Did your run complete successfully?
- Where is your case directory?
- Where is your run directory?
- What sorts of files did you get as output?
- What years did you run over?